In [3]:
import os
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GCNConv, global_mean_pool

from rdkit import Chem

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [17]:
random_state=25


random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Imposto PyTorch in modalità deterministica
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # per determinismo cuBLAS CUDA 11+


In [5]:
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db=client["PolymerPrediction"]
collection=db["biceranoTg"]
target="Tg"
#print(collection.count_documents({}))

docs = list(collection.find({}))
df = pd.DataFrame(docs)

# espando il dizionario "properties"
prop_df = pd.json_normalize(df["properties"])

# integro nel dataframe
df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)
df["smiles"]=df["smiles"].str.replace("[*]", "", regex=False)
# rimuovo righe senza SMILES o target o con SMILES malformato
df = df.dropna(subset=["smiles", target])
df_valid = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
df_valid.describe()

,Tg
count,304.000000
mean,360.565789
std,114.734917
min,130.000000
25%,272.000000
50%,359.000000
75%,418.250000
max,685.000000


## Featurizzazione dei polimeri come grafi 

Qui uso la funzione `smiles2graph` della libreria OGB (Open Graph Benchmark) per
convertire i SMILES dei polimeri in grafi utilizzabili da PyTorch Geometric.

In pratica:
- RDKit legge lo SMILES e costruisce la molecola
- OGB estrae feature per ogni atomo e ogni legame
- ottengo tensori (node_feat, edge_index, edge_feat) pronti per la GNN

In [6]:
from ogb.utils.mol import smiles2graph
from torch_geometric.data import Data
import torch

def mol_to_graph_ogb(smiles: str, y_value: float) -> Data:
    """
    Converte un SMILES in un grafo PyG usando la featurizzazione di OGB.

    Basato su: Open Graph Benchmark (Hu et al., NeurIPS 2020),
    tramite la funzione ogb.utils.mol.smiles2graph.
    """
    g = smiles2graph(smiles)

    # node_feat: array [num_nodes, num_node_features]
    x = torch.tensor(g["node_feat"], dtype=torch.float)

    # edge_index: array [2, num_edges]
    edge_index = torch.tensor(g["edge_index"], dtype=torch.long)

    # edge_attr: feature dei legami
    # edge_attr = torch.tensor(g["edge_feat"], dtype=torch.float)

    y = torch.tensor([y_value], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, y=y)
    return data

In [7]:
class PolymerDataset(InMemoryDataset):
    def __init__(self, dataframe, smiles_col, target_col,
                 root="pyg_polymer_dataset",  # root valida
                 transform=None, pre_transform=None):
        
        self.dataframe = dataframe.reset_index(drop=True)
        self.smiles_col = smiles_col
        self.target_col = target_col
        
        super().__init__(root=root, transform=transform, pre_transform=pre_transform)

        # salta completamente la pipeline {raw → processed}
        self.data, self.slices = self._process_dataframe()

    # ---------- Disattivo completamente la pipeline su disco ----------
    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return []

    def download(self):
        pass

    def process(self):
        pass
    # ------------------------------------------------------------------

    def _process_dataframe(self):
        data_list = []
        for idx, row in self.dataframe.iterrows():
            smiles = row[self.smiles_col]
            y_val = row[self.target_col]
            try:
                data = mol_to_graph_ogb(smiles, y_val)
                data_list.append(data)
            except Exception as e:
                print(f"Errore su riga {idx} ({smiles}): {e}")

        data, slices = self.collate(data_list)
        return data, slices

In [8]:
indices = np.arange(len(df_valid))

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=random_state)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=random_state)

df_train = df_valid.iloc[train_idx].reset_index(drop=True)
df_val   = df_valid.iloc[val_idx].reset_index(drop=True)
df_test  = df_valid.iloc[test_idx].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)


(194, 49, 61)

In [9]:
train_dataset = PolymerDataset(df_train, "smiles", target)
val_dataset   = PolymerDataset(df_val,   "smiles", target)
test_dataset  = PolymerDataset(df_test,  "smiles", target)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Processing...
Done!


Train: 194
Val: 49
Test: 61


Processing...
Done!
Processing...
Done!


In [10]:
g = torch.Generator()
g.manual_seed(random_state)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=g)
val_loader   = DataLoader(val_dataset, batch_size=32, generator=g)
test_loader  = DataLoader(test_dataset, batch_size=32, generator=g)


# Regressors

In [11]:
class GINRegressor(nn.Module):
    def __init__(self, in_channels, h=64, layers=3):
        super().__init__()

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        nn_in = nn.Sequential(nn.Linear(in_channels, h), nn.ReLU(), nn.Linear(h, h))
        self.convs.append(GINConv(nn_in))
        self.bns.append(nn.LayerNorm(h))

        for _ in range(layers-1):
            nn_k = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
            self.convs.append(GINConv(nn_k))
            self.bns.append(nn.LayerNorm(h))

        self.lin1 = nn.Linear(h, h)
        self.lin2 = nn.Linear(h, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)

        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x).view(-1)

In [12]:
class GCNRegressor(nn.Module):
    def __init__(self, in_channels, h=64, layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        # Primo layer GCN: da in_channels a h
        self.convs.append(GCNConv(in_channels, h))
        self.bns.append(nn.LayerNorm(h))

        # Altri layer GCN: da h a h
        for _ in range(layers - 1):
            self.convs.append(GCNConv(h, h))
            self.bns.append(nn.LayerNorm(h))

        self.lin1 = nn.Linear(h, h)
        self.lin2 = nn.Linear(h, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)

        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x).view(-1)

# Model and Optimizer Initialization

In [13]:
in_channels = train_dataset[0].x.size(1)
model = GCNRegressor(in_channels=in_channels, h=64, layers=3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',      # voglio minimizzare la metrica di validazione
    factor=0.5,      # moltiplica il LR per 0.5 quando non migliora
    patience=10,     # dopo 10 epoche senza miglioramenti
)

# Training and Evaluation

In [14]:
def train_epoch(model, loader):
    model.train()
    tot = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(out, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        tot += loss.item() * batch.num_graphs
    return tot / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    ys, preds = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            ys.append(batch.y.cpu().numpy())
            preds.append(out.cpu().numpy())

    ys = np.concatenate(ys)
    preds = np.concatenate(preds)
    return mean_absolute_error(ys, preds), r2_score(ys, preds)

In [15]:
best = None
best_mae = float("inf")
patience = 40
epochs_no_improve = 0

for epoch in range(1, 301):
    loss = train_epoch(model, train_loader)
    val_mae, val_r2 = evaluate(model, val_loader)

    scheduler.step(val_mae)

    if val_mae < best_mae - 1e-4:  # piccolo margine per evitare fluttuazioni numeriche
        best_mae = val_mae
        best = {
            "epoch": epoch,
            "state": model.state_dict(),
            "mae": val_mae,
            "r2": val_r2,
        }
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} "
              f"loss {loss:.3f} "
              f"val MAE {val_mae:.2f} "
              f"val R2 {val_r2:.3f}")

    # Early stopping
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

print("Best:", best)

Epoch 001 loss 141865.811 val MAE 369.04 val R2 -11.527
Epoch 020 loss 108267.618 val MAE 315.21 val R2 -8.409
Epoch 040 loss 27896.810 val MAE 135.38 val R2 -1.289
Epoch 060 loss 7118.667 val MAE 62.04 val R2 0.412
Epoch 080 loss 3840.724 val MAE 49.73 val R2 0.646
Epoch 100 loss 3931.968 val MAE 50.92 val R2 0.637
Epoch 120 loss 3036.218 val MAE 47.34 val R2 0.681
Epoch 140 loss 2960.150 val MAE 47.18 val R2 0.689
Epoch 160 loss 2730.376 val MAE 45.48 val R2 0.724
Epoch 180 loss 2705.131 val MAE 44.68 val R2 0.725
Early stopping at epoch 188
Best: {'epoch': 148, 'state': OrderedDict({'convs.0.bias': tensor([ 0.0894,  0.0319,  0.0619,  0.0513,  0.0173, -0.0579, -0.0417,  0.0278,
         0.0110,  0.0312,  0.0749,  0.0096,  0.0108,  0.0559,  0.0505,  0.0413,
        -0.0224,  0.0341,  0.0605, -0.0003,  0.0072, -0.0128, -0.0077,  0.0149,
         0.0618,  0.0449, -0.0419,  0.0166, -0.0132,  0.0594,  0.0255, -0.0195,
         0.0070,  0.0217,  0.0738,  0.0796,  0.0687,  0.0297, -0.0553, 

In [16]:
model.load_state_dict(best["state"])
test_mae, test_r2 = evaluate(model, test_loader)
print("TEST MAE:", test_mae)
print("TEST R2:", test_r2)


TEST MAE: 45.11917495727539
TEST R2: 0.7316142916679382
